### **BATCH PROCESSING**

### Normal Ingestion

In [0]:
df1 = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferschema", "true")\
    .load("/Volumes/pysparkdbt/source/source_data/customers/")

In [0]:
display(df1)

In [0]:
schema_customers = df1.schema
schema_customers 

### Ingestion using Array

In [0]:
array = [
    {
        "name" : "customers",
        "schema" : "StructType([StructField('customer_id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('phone_number', StringType(), True), StructField('city', StringType(), True), StructField('signup_date', DateType(), True), StructField('last_updated_timestamp', TimestampType(), True)])" 
    }
]

In [0]:
for i in array:
    print(i['schema'])

### **SPARK STREAMING**

In [0]:
entities = ['customers','trips','locations','payments','vehicles','drivers']

In [0]:
for entity in entities: 

    df_batch = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferschema", "true")\
    .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

    schema_entity = df_batch.schema    

    df = spark.readStream.format("csv")\
            .option("header", True)\
            .schema(schema_entity)\
            .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")
    
    df.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation", f"/Volumes/pysparkdbt/bronze/checkpoint/{entity}/")\
            .trigger(once=True)\
            .toTable(f"pysparkdbt.bronze.{entity}")